# Week 5 Laboratory
# Task 1: Image Classification using Multi-Layer Perceptron (MLP)

---

## Learning Objectives

After completing this laboratory, you should be able to:

✓ Extract 3D color histogram features from image datasets using OpenCV.

✓ Split a dataset into **Training (60%)**, **Validation (20%)**, and **Test (20%)** sets.

✓ Evaluate candidate MLP architectures on a validation set to select the optimal hidden layer structure.

✓ Evaluate an MLP classifier using Accuracy, Precision, Recall, F1-score, and Confusion Matrix.

✓ Benchmark model inference speed and build an equivalent MLP using **Keras**.

---
## Setup Environment

Install necessary packages to run this notebook.

In [ ]:
!pip install opencv-python numpy matplotlib tqdm scikit-learn pandas tensorflow

---
## 1. Imports & Pre-defined Helper Functions

We first import required packages and provide four utility functions:
* preprocess_image: Reads and resizes an image to 256x256.
* extract_color_histogram: Computes normalized 3D color histograms (6x6x6 = 216 bins).
* load_dataset: Loads images from flowers/ across all 5 flower categories.
* show_images: Displays correctly or incorrectly classified images in subplots.

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from timeit import default_timer as timer

from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)

import warnings
warnings.filterwarnings('ignore')

LABELS = ['daisy', 'dandelion', 'rose', 'sunflower', 'tulip']

def preprocess_image(path_to_image, img_size=256):
    """Read and resize an input image."""
    img = cv2.imread(path_to_image, cv2.IMREAD_COLOR)
    img = cv2.resize(img, (img_size, img_size))
    return np.array(img)

def extract_color_histogram(dataset, hist_size=6):
    """Extract normalized 3D color histogram features (6x6x6 = 216 bins)."""
    col_hist = []
    for img in dataset:
        hist = cv2.calcHist([img], [0, 1, 2], None, (hist_size, hist_size, hist_size), [0, 256, 0, 256, 0, 256])
        col_hist.append(cv2.normalize(hist, None, 0, 1, cv2.NORM_MINMAX).flatten())
    return np.array(col_hist)

def load_dataset(base_path='flowers'):
    """Load dataset images and labels with automatic directory fallback."""
    if not os.path.exists(base_path):
        if os.path.exists(os.path.join('..', 'Week3', 'flowers')):
            base_path = os.path.join('..', 'Week3', 'flowers')
        elif os.path.exists(os.path.join('Week3', 'flowers')):
            base_path = os.path.join('Week3', 'flowers')
            
    X, Y = [], []
    for i in range(len(LABELS)):
        current_size = len(X)
        dir_path = os.path.join(base_path, LABELS[i])
        if os.path.exists(dir_path):
            for img in tqdm(os.listdir(dir_path), desc=f"Loading {LABELS[i]}"):
                if not img.startswith('.'):
                    X.append(preprocess_image(os.path.join(dir_path, img)))
                    Y.append(LABELS[i])
            print(f'Loaded {len(X) - current_size} {LABELS[i]} images')
    return X, Y

def show_images(ground_truth, predictions, img_test, correct=True, num_images=5):
    """Show correctly or incorrectly classified images in subplots."""
    count = 0
    plt.figure(figsize=(15, 3))
    for i in range(len(ground_truth)):
        if (ground_truth[i] == predictions[i]) == correct:
            count += 1
            plt.subplot(1, num_images, count)
            plt.imshow(cv2.cvtColor(img_test[i], cv2.COLOR_BGR2RGB))
            color = 'darkgreen' if correct else 'crimson'
            plt.title(f"GT: {ground_truth[i]}\nPred: {predictions[i]}", fontsize=9, color=color, fontweight='bold')
            plt.axis('off')
            if count == num_images:
                break
    plt.tight_layout()
    plt.show()

print("Environment set up successfully!")

---
## STEP 1. Load Dataset

Load the flower image dataset into memory using load_dataset().

In [ ]:
# STEP 1. Load dataset
# UNCOMMENT BELOW
# X, Y = load_dataset('flowers')

print(f"Total samples loaded: {len(X)}")

---
## STEP 2. Split Dataset (Train, Validation, Test)

We split the dataset into three subsets:
* **Train Set (60%)**: Fits the neural network.
* **Validation Set (20%)**: Selects the optimal network architecture.
* **Test Set (20%)**: Held-out set for final model evaluation.

### Task:
Use train_test_split twice:
1. Reserve 20% of data for validation (img_val, y_val).
2. Reserve 20% of original data (25% of remaining) for testing (img_test, y_test).

In [ ]:
# STEP 2. Split dataset into train, validation, and test sets
# 1. Split 20% of (X, Y) as validation set (img_val, y_val)
# 2. Split 25% of remaining training data as test set (img_test, y_test)
# YOUR CODE HERE
# img_train, img_val, y_train, y_val = train_test_split(..., ..., test_size=0.2, random_state=1)
# img_train, img_test, y_train, y_test = train_test_split(..., ..., test_size=0.25, random_state=1)

print(f"Training samples  : {len(img_train)} ({len(img_train)/len(X)*100:.1f}%)")
print(f"Validation samples: {len(img_val)} ({len(img_val)/len(X)*100:.1f}%)")
print(f"Test samples      : {len(img_test)} ({len(img_test)/len(X)*100:.1f}%)")

---
## STEP 3. Feature Extraction

Extract 3D color histogram feature vectors (216 dimensions per image) from the training, validation, and test image sets.

### Task:
Apply extract_color_histogram to img_train, img_val, and img_test to produce x_train, x_val, and x_test.

In [ ]:
# STEP 3. Extract colour histogram features from the datasets
# Extract 3D color histograms for img_train, img_val, and img_test
# YOUR CODE HERE
# x_train = extract_color_histogram(...)
# x_val   = extract_color_histogram(...)
# x_test  = extract_color_histogram(...)

print(f"x_train shape: {x_train.shape}")
print(f"x_val shape  : {x_val.shape}")
print(f"x_test shape : {x_test.shape}")

---
## STEP 4. Define Candidate MLP Architectures

Given input size $N_{in} = 216$ and output classes $N_{out} = 5$, we define 3 hidden layer sizes using heuristics:
* **Rule 1 (n1):** $N_{out} < n_1 < N_{in} \implies 100$
* **Rule 2 (n2):** $n_2 = (2/3 \times N_{in}) + N_{out} = 149$
* **Rule 3 (n3):** $n_3 < 2 \times N_{in} \implies 400$

### Task:
Define 9 candidate structures across 1, 2, and 3 hidden layers.

In [ ]:
# STEP 4. Define 9 different structures
# Define hidden layer sizes according to the 3 rules
# YOUR CODE HERE
# n_hidden_1 = ...  # Rule 1: n_output < n_hidden < n_input
# n_hidden_2 = ...  # Rule 2: n_hidden = (2/3 * n_input) + n_output
# n_hidden_3 = ...  # Rule 3: n_hidden < 2 * n_input

# n_hidden_options = [
#     n_hidden_1, n_hidden_2, n_hidden_3,
#     (..., ...), (..., ...), (..., ...),
#     (..., ..., ...), (..., ..., ...), (..., ..., ...)
# ]

for idx, opt in enumerate(n_hidden_options, 1):
    print(f"Option {idx}: {opt}")

---
## STEP 5. Determine the Optimal MLP Structure

Train MLPClassifier for each architecture on (x_train, y_train) and evaluate on (x_val, y_val) to select optimal_n.

In [ ]:
# STEP 5. Determine the optimal structure
val_perform = []

# Loop through n_hidden_options:
# 1. Initialize clf = MLPClassifier(hidden_layer_sizes=test_case, activation='relu', solver='adam', max_iter=1500, random_state=1, early_stopping=True)
# 2. Fit clf on x_train, y_train
# 3. Predict on x_val and calculate val_accuracy using accuracy_score
# 4. Append val_accuracy to val_perform
# YOUR CODE HERE
# for test_case in tqdm(n_hidden_options):
#     clf = MLPClassifier(hidden_layer_sizes=..., activation='relu', solver='adam', max_iter=1500, random_state=1, early_stopping=True)
#     clf.fit(..., ...)
#     predictions = clf.predict(...)
#     acc = accuracy_score(..., predictions)
#     val_perform.append(...)

optimal_idx = np.argmax(val_perform)
optimal_n = n_hidden_options[optimal_idx]
print(f"\nOptimal MLP structure is: {optimal_n} (Val Acc: {val_perform[optimal_idx]*100:.2f}%)")

Let's plot Validation Accuracy vs. MLP architectures.

In [ ]:
# Plot validation performance
plt.figure(figsize=(10, 4.5))
arch_labels = [str(opt) for opt in n_hidden_options]
bars = plt.bar(range(len(n_hidden_options)), [acc * 100 for acc in val_perform], color='steelblue', edgecolor='black', alpha=0.85)
bars[optimal_idx].set_color('crimson')

plt.xticks(range(len(n_hidden_options)), arch_labels, rotation=45, ha='right', fontsize=9)
plt.ylabel('Validation Accuracy (%)', fontsize=10)
plt.title('Validation Accuracy vs. MLP Architecture', fontsize=12, fontweight='bold')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

---
## STEP 6. Train Final MLP Classifier with Optimal Structure

Train the final MLPClassifier using optimal_n on x_train, y_train and record training time.

In [ ]:
# STEP 6. Train final MLP classifier
# Record training duration using timer() and fit MLPClassifier with optimal_n
# YOUR CODE HERE
# start = timer()
# clf = MLPClassifier(hidden_layer_sizes=..., activation='relu', solver='adam', max_iter=1500, random_state=1, early_stopping=True)
# clf.fit(..., ...)
# end = timer()

print(f"Training completed in {end - start:.4f} seconds")

---
## STEP 7. Benchmark Inference Speed on Test Set

Evaluate predictions on x_test and measure the average inference latency per image.

In [ ]:
# STEP 7. Benchmark inference speed
# Measure time taken to predict on test dataset
# YOUR CODE HERE
# start = timer()
# y_pred = clf.predict(...)
# end = timer()

latency = (end - start) / len(x_test)
print(f"Inference latency: {latency:.6f} s/image ({latency * 1000:.3f} ms/image)")

---
## STEP 8. Report Classification Metrics

Calculate Accuracy, Precision, Recall, and F1-score on test set predictions.

In [ ]:
# STEP 8. Report classification metrics
# Calculate accuracy_score, precision_score, recall_score, and f1_score on (y_test, y_pred)
# For precision, recall, and f1, use average='macro'
# YOUR CODE HERE
# accuracy  = accuracy_score(y_test, y_pred)
# precision = precision_score(..., ..., average='macro')
# recall    = recall_score(..., ..., average='macro')
# f1        = f1_score(..., ..., average='macro')

print(f"Accuracy  : {accuracy * 100:.2f}%")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-score  : {f1:.4f}\n")
print(classification_report(y_test, y_pred, target_names=LABELS))

---
## STEP 9. Plot Confusion Matrix

Display the confusion matrix for test predictions.

In [ ]:
# STEP 9. Plot confusion matrix
# UNCOMMENT BELOW
# ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=LABELS, cmap='Blues', xticks_rotation=45)
# plt.title('Confusion Matrix - Scikit-Learn MLP', fontweight='bold')
# plt.show()

---
## STEP 10. Show Classified Images

Display top 5 correctly classified images and top 5 incorrectly classified images using show_images.

In [ ]:
# STEP 10. Show correctly and incorrectly classified images
# UNCOMMENT BELOW
# show_images(ground_truth=y_test, predictions=y_pred, img_test=img_test, correct=True, num_images=5)
# show_images(ground_truth=y_test, predictions=y_pred, img_test=img_test, correct=False, num_images=5)

---
## STEP 11. Alternative Implementation: MLP in Keras

Implement the equivalent MLP in **Keras** using Sequential and Dense layers (Slide 16).

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder

# Encode labels to one-hot vectors
label_encoder = LabelEncoder()
y_train_cat = to_categorical(label_encoder.fit_transform(y_train), 5)
y_val_cat   = to_categorical(label_encoder.transform(y_val), 5)
y_test_cat  = to_categorical(label_encoder.transform(y_test), 5)

# STEP 11. Build, compile and train Keras MLP
# 1. Build Sequential model: Dense(128, 'relu') -> Dense(64, 'relu') -> Dense(5, 'softmax')
# 2. Compile model with optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy']
# 3. Train with model.fit(x_train, y_train_cat, validation_data=(x_val, y_val_cat), epochs=30, batch_size=32, verbose=1)
# YOUR CODE HERE
# keras_mlp = Sequential([
#     Dense(128, activation="relu", input_shape=(...,)),
#     Dense(64, activation=...),
#     Dense(..., activation="softmax")
# ])
# keras_mlp.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
# history = keras_mlp.fit(..., ..., validation_data=(..., ...), epochs=30, batch_size=32, verbose=1)

Plot Keras training curves and evaluate on test set.

In [ ]:
# Plot Keras training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(history.history['accuracy'], label='Train Acc', lw=2)
ax1.plot(history.history['val_accuracy'], label='Val Acc', lw=2, linestyle='--')
ax1.set_title('Keras MLP Accuracy', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history.history['loss'], label='Train Loss', lw=2)
ax2.plot(history.history['val_loss'], label='Val Loss', lw=2, linestyle='--')
ax2.set_title('Keras MLP Loss', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

test_loss, test_acc = keras_mlp.evaluate(x_test, y_test_cat, verbose=0)
print(f"Keras MLP Test Accuracy : {test_acc * 100:.2f}%")
print(f"Scikit-Learn Test Acc   : {accuracy * 100:.2f}%")

---
## Reflection Questions

1. **Q1:** How does the MLP's training and inference speed compare to KNN in Week 3?
2. **Q2:** What is the purpose of the ReLU activation function?
3. **Q3:** Why are 3D color histograms limited for complex image recognition, and how will CNNs solve this?